In [1]:
import torch
import numpy as np
import gpytorch
import pandas as pd
from matplotlib import pyplot as plt
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from deep_gp.gptorch_example import ExactGPModel, RBFLinearGPModel
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
# load data
features = pd.read_csv('../features.csv')
targets = pd.read_csv('../targets.csv')

data = features.merge(targets[['study_id','case_ISUP']], on='study_id')

X = data.drop(['study_id','patient_id','case_ISUP'], axis=1)
y = data['case_ISUP']


In [3]:
df = X.copy()
df['label'] = y

# target per class
N = len(df)
T = N // 6
print("Target per class:", T)

# fill minority classes using class 0
idx_class0 = df[df['label'] == 0].index
used = set()

for cls in range(1, 6):
    current = (df['label'] == cls).sum()
    needed = T - current
    print(f"Class {cls} needs {needed}")
    
    if needed <= 0:
        continue
    
    cls_idx = df[df['label'] == cls].index
    knn = NearestNeighbors(n_neighbors=1)
    knn.fit(df.loc[cls_idx, X.columns])
    
    distances, indices = knn.kneighbors(df.loc[idx_class0, X.columns])
    sorted_idx = np.argsort(distances.flatten())
    
    selected = []
    for i in sorted_idx:
        idx0 = idx_class0[i]
        if idx0 not in used:
            selected.append(idx0)
            used.add(idx0)
        if len(selected) == needed:
            break
    
    df.loc[selected, 'label'] = cls

print("\nFinal balanced counts:")
print(df['label'].value_counts().sort_index())

X_balanced = df.drop(columns=['label'])
y_balanced = df['label']


Target per class: 171
Class 1 needs 14
Class 2 needs 17
Class 3 needs 102
Class 4 needs 144
Class 5 needs 136

Final balanced counts:
label
0    176
1    171
2    171
3    171
4    171
5    171
Name: count, dtype: int64


## Usage of ScaleKernel(RBFKernel())

In [4]:
# scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_balanced_scaled = scaler.fit_transform(X_balanced)

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced_scaled, y_balanced, test_size=0.2, random_state=42
)

# convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)



In [5]:
# train GP

likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(X_train_tensor, y_train_tensor, likelihood)

smoke_test = ('CI' in os.environ)
training_iter = 2 if smoke_test else 500

model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)


In [6]:
for i in range(training_iter):
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = -mll(output, y_train_tensor)
    loss.backward()
    
    print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
        model.covar_module.base_kernel.lengthscale.item(),
        model.likelihood.noise.item()
    ))
    
    optimizer.step()


Iter 1/500 - Loss: 4.369   lengthscale: 0.693   noise: 0.693
Iter 2/500 - Loss: 4.012   lengthscale: 0.744   noise: 0.744
Iter 3/500 - Loss: 3.704   lengthscale: 0.798   noise: 0.798
Iter 4/500 - Loss: 3.438   lengthscale: 0.853   noise: 0.853
Iter 5/500 - Loss: 3.210   lengthscale: 0.910   noise: 0.909
Iter 6/500 - Loss: 3.013   lengthscale: 0.969   noise: 0.966
Iter 7/500 - Loss: 2.844   lengthscale: 1.031   noise: 1.023
Iter 8/500 - Loss: 2.698   lengthscale: 1.095   noise: 1.079
Iter 9/500 - Loss: 2.571   lengthscale: 1.162   noise: 1.135
Iter 10/500 - Loss: 2.462   lengthscale: 1.232   noise: 1.189
Iter 11/500 - Loss: 2.368   lengthscale: 1.304   noise: 1.242
Iter 12/500 - Loss: 2.287   lengthscale: 1.381   noise: 1.292
Iter 13/500 - Loss: 2.219   lengthscale: 1.460   noise: 1.341
Iter 14/500 - Loss: 2.162   lengthscale: 1.541   noise: 1.386
Iter 15/500 - Loss: 2.113   lengthscale: 1.625   noise: 1.429
Iter 16/500 - Loss: 2.075   lengthscale: 1.711   noise: 1.469
Iter 17/500 - Los

In [7]:
# evaluation

model.eval()
likelihood.eval()

f_preds = model(X_test_tensor)
y_preds = likelihood(model(X_test_tensor))

f_mean = f_preds.mean


In [8]:
# metrics 

mse = mean_squared_error(
    y_test_tensor.detach().numpy(),
    f_mean.detach().numpy()
)

r2 = r2_score(
    y_test_tensor.detach().numpy(),
    f_mean.detach().numpy()
)

print("MSE:", mse)
print("R²:", r2)


MSE: 2.290980100631714
R²: 0.22471803426742554


In [9]:
f_std = y_preds.stddev  # standard deviation = sqrt(variance)


In [10]:
# calculation of uncertainty 

df_unc = pd.DataFrame({
    "true": y_test_tensor.detach().cpu().numpy(),
    "std": f_std.detach().cpu().numpy()
})
print(df_unc.groupby("true")["std"].mean())

true
0.0    1.570668
1.0    1.556045
2.0    1.544730
3.0    1.547083
4.0    1.547605
5.0    1.559631
Name: std, dtype: float32


## Usage of RBF + Linear kernel

In [11]:
# knn balancing

df = X.copy()
df['label'] = y

N = len(df)
T = N // 6
print("Target per class:", T)

idx_class0 = df[df['label'] == 0].index
used = set()

for cls in range(1, 6):
    current = (df['label'] == cls).sum()
    needed = T - current
    print(f"Class {cls} needs {needed}")
    
    if needed <= 0:
        continue
    
    cls_idx = df[df['label'] == cls].index
    knn = NearestNeighbors(n_neighbors=1)
    knn.fit(df.loc[cls_idx, X.columns])
    
    distances, indices = knn.kneighbors(df.loc[idx_class0, X.columns])
    sorted_idx = np.argsort(distances.flatten())
    
    selected = []
    for i in sorted_idx:
        idx0 = idx_class0[i]
        if idx0 not in used:
            selected.append(idx0)
            used.add(idx0)
        if len(selected) == needed:
            break
    
    df.loc[selected, 'label'] = cls

print("\nFinal balanced counts:")
print(df['label'].value_counts().sort_index())

X_balanced = df.drop(columns=['label'])
y_balanced = df['label']

# scaling

scaler = StandardScaler()
X_balanced_scaled = scaler.fit_transform(X_balanced)


# train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced_scaled, y_balanced, test_size=0.2, random_state=42
)


# convvert to tensors

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)


# train GP

likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = RBFLinearGPModel(X_train_tensor, y_train_tensor, likelihood)

training_iter = 500
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = -mll(output, y_train_tensor)
    loss.backward()
    
    if (i+1) % 50 == 0:
        print(f"Iter {i+1}/{training_iter} - Loss: {loss.item():.3f} "
              f"noise: {model.likelihood.noise.item():.3f}")
    
    optimizer.step()


# evaluation

model.eval()
likelihood.eval()

with torch.no_grad():
    preds = likelihood(model(X_test_tensor))

f_mean = preds.mean
f_var = preds.variance
f_std = preds.stddev


Target per class: 171
Class 1 needs 14
Class 2 needs 17
Class 3 needs 102
Class 4 needs 144
Class 5 needs 136

Final balanced counts:
label
0    176
1    171
2    171
3    171
4    171
5    171
Name: count, dtype: int64
Iter 50/500 - Loss: 3.024 noise: 0.936
Iter 100/500 - Loss: 2.299 noise: 1.094
Iter 150/500 - Loss: 2.042 noise: 1.123
Iter 200/500 - Loss: 2.000 noise: 1.128
Iter 250/500 - Loss: 1.966 noise: 1.135
Iter 300/500 - Loss: 1.978 noise: 1.146
Iter 350/500 - Loss: 1.943 noise: 1.156
Iter 400/500 - Loss: 1.943 noise: 1.181
Iter 450/500 - Loss: 1.928 noise: 1.225
Iter 500/500 - Loss: 1.926 noise: 1.280


In [12]:
# metrics

mse = mean_squared_error(y_test_tensor.numpy(), f_mean.numpy())
r2 = r2_score(y_test_tensor.numpy(), f_mean.numpy())

print("\n--- GP Performance (RBF + Linear Kernel) ---")
print("MSE:", mse)
print("R²:", r2)


# uncertainty per class
df_unc = pd.DataFrame({
    "true": y_test_tensor.numpy(),
    "std": f_std.numpy()
})

print("\n--- Predictive Uncertainty per ISUP Class ---")
print(df_unc.groupby("true")["std"].mean())



--- GP Performance (RBF + Linear Kernel) ---
MSE: 2.482182502746582
R²: 0.16001397371292114

--- Predictive Uncertainty per ISUP Class ---
true
0.0    1.551928
1.0    1.505285
2.0    1.477468
3.0    1.482480
4.0    1.465848
5.0    1.497028
Name: std, dtype: float32
